# Bigkas FER v2: training walkthrough

This notebook trains the Bigkas facial expression model (**confused, engaged, frustrated, neutral**) on **your own** dataset, split by participant so the test only uses people the model has never seen.

**How to use it:** run the cells **from top to bottom**. Click a cell, then press **▶** (or **Shift + Enter**). Wait until a cell finishes (the ▶ stops spinning and shows a number) before running the next.

**Before you start:**
1. **Runtime → Change runtime type → T4 GPU → Save** (faster; optional).
2. Have these files ready on your computer:
   - `fer_dataset.zip`, your dataset zipped (`participants.csv` + `P01/`, `P02/`, … folders)
   - `train_fer_v2.py`, `audit_fer_dataset.py`, `verify_fer_model.py`, from `Bigkas\scripts\fer_training\`

**Keep this tab open while it runs.** Colab deletes everything when the session disconnects.

Red lines that mention *cuda, cuDNN, TensorRT, protobuf* or *Python version 3.11* are only warnings; ignore them.

## Step 1 — Set up Python 3.11 with TensorFlow
Colab's own Python is too new for this TensorFlow version, so this makes a separate Python 3.11 setup in `/content/fer`. Takes about **3–5 minutes**.

In [ ]:
!pip install -q uv
!uv venv -q -p 3.11 /content/fer
!uv pip install -q --python /content/fer/bin/python "tensorflow[and-cuda]==2.15.1" "numpy<2" pandas scikit-learn matplotlib pillow
!/content/fer/bin/python -c "import tensorflow as tf; print('TensorFlow', tf.__version__, '· GPU:', bool(tf.config.list_physical_devices('GPU')))"

**Check:** the last line should say `TensorFlow 2.15.1`. `GPU: True` means the T4 is being used; `False` still works, just slower.

## Step 2 — Upload your files
A **Choose Files** button appears below the cell. Select **all four files** at once (hold **Ctrl**):
`fer_dataset.zip`, `train_fer_v2.py`, `audit_fer_dataset.py`, `verify_fer_model.py`.

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded))

## Step 3 — Unzip the dataset and find it
This unzips your dataset and finds the folder that contains `participants.csv`, even if the ZIP has an extra folder inside.

In [ ]:
import pathlib, subprocess
subprocess.run(["unzip", "-q", "-o", "fer_dataset.zip", "-d", "dataset_unzipped"], check=True)
found = sorted(pathlib.Path("dataset_unzipped").rglob("participants.csv"))
if not found:
    raise SystemExit("❌ No participants.csv inside fer_dataset.zip — check the ZIP")
DATA_DIR = str(found[0].parent)
print("✅ Dataset folder:", DATA_DIR)
print(sorted(p.name for p in found[0].parent.iterdir())[:20])

**Check:** the list should show `participants.csv` and your participant folders (`P01`, `P02`, …).

## Step 4 — Audit the dataset (before training)
Checks that `participants.csv` matches the folders, the four state folders are spelled right, each eyelid group has at least 3 people, every image opens, and no photo is filed under two people.

**It must end with ✅.** If it shows ❌ errors: fix them on your computer, zip the dataset again, re-run **Step 2** (upload only the new ZIP), then Steps 3 and 4.

In [ ]:
!/content/fer/bin/python audit_fer_dataset.py --data-dir "$DATA_DIR"

## Step 5 — Train
Two phases:
1. **Head only**: MobileNetV2 frozen, up to 15 epochs.
2. **Fine-tune**: top 30 layers unfrozen at learning rate 1e-5, up to 25 epochs.

Early stopping ends each phase when validation stops improving. Takes **10–40 minutes**.

At the end it prints accuracy on the **unseen test participants**, overall and for monolid vs non-monolid.

In [ ]:
!MPLBACKEND=Agg /content/fer/bin/python train_fer_v2.py --data-dir "$DATA_DIR"

## Step 6 — Look at the results

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("fer_v2_output/evaluation_report.md", encoding="utf-8").read()))
display(Image("fer_v2_output/confusion_matrices.png"))

**How to read it:**
- **Balanced accuracy** is the fairest single number (25% = guessing).
- **Gap (non − mono)**: a positive number means the model does worse on monolid participants.
- In the confusion matrices, the **diagonal** is correct answers. A row whose numbers sit mostly *off* the diagonal (for example, confused photos predicted as neutral) is a state the model is failing on.
- Each group's result is only as reliable as the number of test people (**n**) shown with it.

## Step 7 — Check the model is ready for the app
**It must end with ✅ Ready.**

In [ ]:
!/content/fer/bin/python verify_fer_model.py --model-dir fer_v2_output

## Step 8 — Download the results
Downloads four files to your computer's **Downloads** folder. If the browser asks, allow multiple downloads.

In [ ]:
from google.colab import files
for name in ["fer_bigkas.tflite", "model_metadata.json", "evaluation_report.md", "confusion_matrices.png"]:
    files.download(f"fer_v2_output/{name}")

## After downloading
1. Copy `fer_bigkas.tflite` into `C:\Users\denve\React\Bigkas\assets\models\` and **replace** the old file. It should be about 9 MB.
2. Put `evaluation_report.md` and `confusion_matrices.png` in `Bigkas\scripts\fer_training\fer_v2_output\` for your paper.
3. In the project: `npx expo start -c --dev-client`, fully close and reopen the app, and test on **FER Training Data → 📸 Collect Photos Now**.

**Don't commit the dataset or the photos to GitHub.** They're people's faces.